# Module 03-2: 3노드 그래프 + Mermaid 시각화 + Stream 실행 (솔루션)

## 학습 목표

1. **FakeLLM 활용**: `common.fake_llm.FakeLLM`을 노드 내에서 사용하는 패턴을 익힌다
2. **3노드 분석 파이프라인**: analyze → summarize → format 흐름을 구현한다
3. **Mermaid 시각화**: `get_graph().draw_mermaid()`로 그래프 구조를 시각화한다
4. **Stream 실행**: `stream()` 메서드로 노드별 중간 상태를 확인한다
5. **invoke vs stream 비교**: 두 실행 방식의 차이를 이해한다

---
> 참고 문서: `docs/part2-first-agent/03-jupyter-dev-environment.md`

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from common.fake_llm import FakeLLM

print("환경 설정 완료")

## 실습 1: TextAnalysisState 정의 및 FakeLLM 초기화 (솔루션)

In [ ]:
# 솔루션
class TextAnalysisState(TypedDict):
    text: str
    analysis: str
    summary: str
    final_report: str

llm = FakeLLM(
    responses={
        "분석": "키워드 분석 완료: 기술, AI, 에이전트 관련 내용",
        "요약": "핵심 요약: LangGraph로 에이전트를 구축하는 방법"
    }
)

print("TextAnalysisState와 FakeLLM 설정 완료")

In [ ]:
# 검증 셀
assert 'TextAnalysisState' in dir(), "TextAnalysisState가 정의되어야 합니다"
assert 'llm' in dir(), "llm 변수가 정의되어야 합니다"
assert isinstance(llm, FakeLLM), "llm은 FakeLLM 인스턴스여야 합니다"
assert len(llm.responses) >= 2, "최소 2개의 패턴-응답이 필요합니다"
print("✅ 실습 1 완료!")

## 실습 2: 3개 노드 함수 구현 (솔루션)

In [ ]:
# 솔루션
def analyze(state: TextAnalysisState) -> dict:
    prompt = f"다음 텍스트를 분석해주세요: {state['text']}"
    response = llm.invoke(prompt)
    return {"analysis": response.content}


def summarize(state: TextAnalysisState) -> dict:
    prompt = f"다음 분석 결과를 요약해주세요: {state['analysis']}"
    response = llm.invoke(prompt)
    return {"summary": response.content}


def format_report(state: TextAnalysisState) -> dict:
    report = (
        f"=== 텍스트 분석 보고서 ===\n"
        f"원본: {state['text']}\n"
        f"분석: {state['analysis']}\n"
        f"요약: {state['summary']}"
    )
    return {"final_report": report}


print("노드 함수 구현 완료")

In [ ]:
# 검증 셀
test_state = {"text": "LangGraph 분석 테스트", "analysis": "", "summary": "", "final_report": ""}

r_analyze = analyze(test_state)
assert "analysis" in r_analyze, "analyze는 'analysis' 키를 반환해야 합니다"
assert len(r_analyze["analysis"]) > 0, "analysis가 비어있지 않아야 합니다"

test_state["analysis"] = r_analyze["analysis"]
r_summarize = summarize(test_state)
assert "summary" in r_summarize, "summarize는 'summary' 키를 반환해야 합니다"

test_state["summary"] = r_summarize["summary"]
r_format = format_report(test_state)
assert "final_report" in r_format, "format_report는 'final_report' 키를 반환해야 합니다"
assert "분석" in r_format["final_report"] or "원본" in r_format["final_report"], \
    "final_report에 분석 내용이 포함되어야 합니다"
print("✅ 실습 2 완료!")

## 실습 3: 그래프 조립 및 Mermaid 시각화 (솔루션)

In [ ]:
# 솔루션
graph = StateGraph(TextAnalysisState)
graph.add_node("analyze", analyze)
graph.add_node("summarize", summarize)
graph.add_node("format_report", format_report)

graph.add_edge(START, "analyze")
graph.add_edge("analyze", "summarize")
graph.add_edge("summarize", "format_report")
graph.add_edge("format_report", END)

app = graph.compile()

mermaid_code = app.get_graph().draw_mermaid()
print("Mermaid 코드:")
print(mermaid_code)
print("그래프 조립 및 시각화 완료")

In [ ]:
# 검증 셀
assert 'app' in dir(), "app 변수가 정의되어야 합니다"
mermaid_code = app.get_graph().draw_mermaid()
assert "analyze" in mermaid_code, "Mermaid 코드에 analyze 노드가 있어야 합니다"
assert "summarize" in mermaid_code, "Mermaid 코드에 summarize 노드가 있어야 합니다"
assert "format_report" in mermaid_code, "Mermaid 코드에 format_report 노드가 있어야 합니다"
print("✅ 실습 3 완료!")

## 실습 4: Stream 실행으로 중간 상태 확인 (솔루션)

In [ ]:
# 솔루션
initial_state = {"text": "LangGraph를 배우는 중입니다", "analysis": "", "summary": "", "final_report": ""}

print("=== Stream 실행 ===")
for chunk in app.stream(initial_state):
    node_name = list(chunk.keys())[0]
    print(f"[{node_name}] 실행됨")
    for key, val in chunk[node_name].items():
        print(f"  {key}: {str(val)[:60]}")

print("\n=== Invoke 실행 ===")
result = app.invoke(initial_state)
print(f"최종 보고서:\n{result['final_report']}")

In [ ]:
# 검증 셀 - invoke와 stream 결과 비교
initial = {"text": "LangGraph 검증 테스트", "analysis": "", "summary": "", "final_report": ""}

invoke_result = app.invoke(initial)
stream_chunks = list(app.stream(initial))
assert len(stream_chunks) == 3, f"stream은 3개의 청크를 반환해야 합니다. 실제: {len(stream_chunks)}"

node_names = [list(chunk.keys())[0] for chunk in stream_chunks]
assert "analyze" in node_names, "stream에 analyze 청크가 있어야 합니다"
assert "summarize" in node_names, "stream에 summarize 청크가 있어야 합니다"
assert "format_report" in node_names, "stream에 format_report 청크가 있어야 합니다"

print(f"invoke 최종 보고서:\n{invoke_result['final_report']}")
print(f"\nstream 청크 수: {len(stream_chunks)}")
print("✅ 실습 4 완료!")

## 정리

### 오늘 배운 내용
- `FakeLLM`을 노드 내에서 사용하여 **API 없이 LLM 호출 시뮬레이션**
- `get_graph().draw_mermaid()`로 **그래프 구조 시각화**
- `stream()` vs `invoke()`: 중간 상태 확인 vs 최종 결과만 반환

### 다음 모듈 안내
**Module 04: 첫 에이전트** - 에러 처리와 조건부 분기가 포함된 실용적인 에이전트를 구축합니다.